[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/quadric_collision/quadric_collision.ipynb)

# Collision of Two Ellipses

This example decides whether two ellipses are apart, touching or overlapping, with one number. Between the two shapes runs a family of blends, a little of one and the rest of the other, from the first to the second. If the ellipses are apart, some blend in between is a hyperbola whose two branches part them, and a line between them separates the pair; if they overlap, every blend in between stays an ellipse bridging the two. One number along the blends, peaking somewhere between, tells which: above zero apart, zero touching, below zero overlapping. At the touching pose that blend gives the contact itself: the shared tangent line and the point where both ellipses meet it.

Points are antivectors and lines vectors, and an ellipse is held by its tangent lines, a dual quadric that maps each line to a point, `Point <- Line`: zero on the lines that touch it, `line & dual(line) == 0`, and sending each tangent to its point of contact. A motor carries the whole map, `motor >> dual(motor << Line)`. Blends of two such maps keep their common tangents, and the number is the blend's determinant, a cubic in how far the blend has gone: four samples fix it, and only finding its peak leaves the algebra. At contact the blend at the peak is singular, and its null line is the shared tangent.

In matrix notation the dual ellipses read as symmetric $3\times3$ matrices $C_1^*$ and $C_2^*$, the blends as the pencil $(1-\lambda)C_1^* + \lambda C_2^*$, and the test as the sign of the maximum of $\det$ over the pencil.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from examples.animation import save_figure
from examples.quadrics.quadric_collision import render
from examples.quadrics.quadric_collision.core import Line, Motor, Point, Quadric, Scalar, along_x, along_y, infinity, mv, origin

np.set_printoptions(precision=4, suppress=True)

## 1. An ellipse held by its tangents

An ellipse's dual vanishes on exactly the lines that touch it, and sends each of them to its point of contact. Of the lines facing a given way, two touch the ellipse, one either side of its centre; the outward one is its tangent that way, whatever the scale or sign of the dual.

In [ ]:
def ellipse(semi_x: float, semi_y: float) -> Quadric:
    """The dual ellipse with the given semi-axes along x and y, centred at the origin: each axis
    direction paired with itself and weighted by its squared semi-axis, less the origin paired with
    itself."""
    return along_x * (Line & along_x) * semi_x**2 + along_y * (Line & along_y) * semi_y**2 - origin * (Line & origin)


def motor(shift_x: float, shift_y: float, angle: float) -> Motor:
    """The rigid motion turning by an angle about the origin, then moving it by the given shifts."""
    translator = (infinity.wedge(mv.x * shift_x + mv.y * shift_y) * -0.5).exp()
    rotor = (mv.xy * (-angle / 2.0)).exp()
    return translator * rotor


def tangent_line(dual: Quadric, normal: Line) -> Line:
    """The tangent line of a dual ellipse with the given outward normal, whatever its scale or sign.

    The lines with the normal's direction are `normal - infinity * offset`. Tangency is quadratic in the
    offset: `a * offset**2 - 2 * b * offset + c == 0`, with `a`, `b` and `c` the pairings of the line at
    infinity and the normal through the dual. Its two roots lie either side of the centre's offset
    `b / a`; the outward one lies beyond it along the normal. Rescaling the dual rescales all three
    coefficients alike, so neither root moves.
    """
    unit = normal.normalized()                                                 # [...] Line
    a, b, c = infinity & dual(infinity), unit & dual(infinity), unit & dual(unit)   # [...] Scalar each
    return unit - infinity * (b / a + ((b * b - a * c) / (a * a)).square_root())


# An ellipse's semi-axes, the shift and turn that place it, and six ways its tangents face.
semi_axes, shift, placement_turn = (2.0, 1.2), (1.5, -0.8), np.radians(35)
facing = np.radians([0, 45, 90, 135, 210, 315])

placement = motor(*shift, placement_turn)
dual = placement >> ellipse(*semi_axes)(placement << Line)                     # [] Point <- Line
# Its tangent lines facing each way: each contains its own point of contact.
normals = (mv.xy * (-facing / 2)).exp() >> mv.x                                # [ways] Line
tangents = tangent_line(dual, normals)                                         # [ways] Line

In [ ]:
print("each tangent paired with its point of contact:", (tangents & dual(tangents)).to_array())

## 2. Three poses, one number

The second ellipse is turned and moved so that its tangent point with one normal lands on the first ellipse's tangent point with the opposite normal: touching. From there it is moved apart along the normal, and into the first. For each pose, the determinant of the blend is a cubic in the blend's share, fixed by four samples, and its peak between zero and one classifies the pose.

In [ ]:
# The first ellipse: semi-axes, centre and turn; the second: semi-axes and turn; the first's normal at the
# contact; and the second's offsets along it, apart, touching and overlapping.
first_semi, first_centre, first_turn = (2.0, 1.0), (-1.2, 0.0), np.radians(25)
second_semi, second_turn = (1.6, 0.9), np.radians(-35)
contact_normal = np.radians(22)
offsets = np.array([0.8, 0.0, -0.6])
# The shares at which the blend's determinant, a cubic, is sampled to fix it; the shares it is drawn at;
# and the share of the blended ellipse drawn between the touching pair.
samples = [[0.0], [1.0], [2.0], [-1.0]]
n_shares = 500
drawn_share = 0.45

In [ ]:
def blend(first: Quadric, second: Quadric, share: Scalar) -> Quadric:
    """The blend of two dual quadrics, the given share of the way from the first to the second."""
    return first * (1.0 - share) + second * share


def cubic_peak(values: Scalar) -> tuple[Scalar, Scalar]:
    """The maximum of a cubic in the share, sampled at 0, 1, 2 and -1 along the first axis.

    The cubic is concave between 0 and 1, so its maximum is the root of the derivative
    `3 * c3 * share**2 + 2 * c2 * share + c1` where the second derivative is negative. Written as
    `c1 / ((c2 * c2 - 3 * c3 * c1).square_root() - c2)`, that root stays finite as c3 vanishes.
    """
    y0, y1, y2, y3 = values
    c3 = (3.0 * y0 - 3.0 * y1 + y2 - y3) / 6.0
    c2 = -y0 + 0.5 * y1 + 0.5 * y3
    c1 = -0.5 * y0 + y1 - y2 / 6.0 - y3 / 3.0
    c0 = y0
    peak = c1 / ((c2 * c2 - 3.0 * c3 * c1).square_root() - c2)
    return peak, ((c3 * peak + c2) * peak + c1) * peak + c0


placement = motor(*first_centre, first_turn)
fixed = placement >> ellipse(*first_semi)(placement << Line)                   # [] Point <- Line
normal = (mv.xy * (-contact_normal / 2)).exp() >> mv.x                         # [] Line
shape = ellipse(*second_semi)                                                  # [] Point <- Line
turn = motor(0.0, 0.0, second_turn)
turned = turn >> shape(turn << Line)                                           # [] Point <- Line
# The two contact points, at unit weight, and the motion that lays the one on the other.
target = fixed(tangent_line(fixed, normal))                                    # [] Point
start = turned(tangent_line(turned, -normal))                                  # [] Point
displacement = target / (target & infinity) - start / (start & infinity)       # [] Point
touch = (infinity.wedge(displacement.dual()) * -0.5).exp() * turn              # [] Motor
poses = (infinity.wedge(normal * offsets) * -0.5).exp() * touch                # [poses] Motor
moved = poses >> shape(poses << Line)                                          # [poses] Point <- Line
# Four samples of the blend's determinant fix the cubic.
parameter, maximum = cubic_peak(blend(fixed, moved, mv.scalar(samples)[:, None]).dual().det())   # [poses] Scalar each

In [ ]:
print("share at the peak, per pose:", parameter.to_array())
print("peak determinant, apart, touching and overlapping:", maximum.to_array())

## 3. The contact, and the shapes between

At the touching pose the blend at its peak is singular: its last singular vector is a line it sends to nothing, the tangent shared by both ellipses, and each ellipse sends it to the same point, the contact. In the figure, apart (top left), the peak blend is a hyperbola, green, whose branches part the two, and the midline between their facing tangents separates them; touching (top right), a blend between is an ellipse, purple, through the contact, red, on the shared tangent; overlapping (bottom left), the peak blend bridges the overlap. Bottom right, the three determinants against the blend's share, each with its peak.

In [ ]:
_, _, lines = blend(fixed, moved[1], parameter[1]).dual().svd()
contact_line = lines[-1].normalized()                                          # [] Line
contact_point = fixed(contact_line).normalized()                               # [] Point
# For the figure: the determinant along the blends, the facing tangents and their midlines, and the blends drawn.
shares = mv.scalar(np.linspace(0.001, 0.999, n_shares)[:, None])                    # [shares] Scalar
determinants = blend(fixed, moved, shares[:, None]).dual().det()               # [shares, poses] Scalar
first = tangent_line(fixed, normal)                                            # [] Line
second = tangent_line(moved, -normal)                                          # [poses] Line
midline = ((first - second) * 0.5).normalized()                                # [poses] Line
separating = blend(fixed, moved[0], parameter[0]).inverse()                    # [] Line <- Point
deforming = blend(fixed, moved[1], drawn_share)                                       # [] Point <- Line
bridging = blend(fixed, moved[2], parameter[2])                                # [] Point <- Line
save_figure(render.draw_collision(fixed, moved, parameter, maximum, shares, determinants, first, second, midline,
                                  separating, deforming, bridging, contact_line, contact_point), "quadric_collision_2d");

In [ ]:
print("the shared tangent against each ellipse:", (contact_line & fixed(contact_line)).to_array(), (contact_line & moved[1](contact_line)).to_array())
print("the second ellipse's contact point against the first's:", (moved[1](contact_line).normalized() & contact_point).norm().to_array())

In [ ]:
# checks
# Every tangent contains its point of contact; the peak determinant is positive apart, zero touching and
# negative overlapping; the shared tangent touches both ellipses, at one point.
np.testing.assert_allclose((tangents & dual(tangents)).to_array(), 0.0, atol=1e-10)
assert maximum[0].to_array() > 0 > maximum[2].to_array()
np.testing.assert_allclose(maximum[1].to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((contact_line & fixed(contact_line)).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((moved[1](contact_line).normalized() & contact_point).norm().to_array(), 0.0, atol=1e-12)